In [ ]:
from NN_model import BinaryClassifier 
from NN_model import ImprovedNN 

In [ ]:
import torch
import torch as nn
from pathlib import Path

def load_model_from_ckpt(ckpt_path, model_cls, input_size, device="cpu"):
    ckpt = torch.load(ckpt_path, map_location=device)

    model = model_cls(
        input_size=input_size,
        hidden_layers=ckpt("hidden_layers"),
        dropout_rate=ckpt("dropout_rate"),
    ).to(device)

    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    return model

In [ ]:
import numpy as np

@torch.no_grad()
def predict_weighted_mp(
    x_features_1d,                  # numpy array shape (n_features,)
    clf_model,                      # BinaryClassifier
    reg_lowint_model,               # RegressorNN
    reg_high_model,                 # RegressorNN
    device="cpu",
):
    # shape (1, n_features)
    x = torch.tensor(x_features_1d, dtype=torch.float32, device=device).unsqueeze(0)

    # ----- classifier -----
    logit_high = clf_model(x)                     # shape (1,)
    prob_high  = torch.sigmoid(logit_high).item() # scalar
    prob_lowint = 1.0 - prob_high

    # ----- regressors -----
    mp_lowint = reg_lowint_model(x).item()
    mp_high   = reg_high_model(x).item()

    # ----- weighted average -----
    mp_weighted = prob_lowint * mp_lowint + prob_high * mp_high

    return {"mp_weighted": mp_weighted}


In [ ]:
import joblib
import pandas as pd

device = "cpu"  # or "cuda" if you trained on GPU and want inference on GPU

# 1) Define feature order EXACTLY as training
# If you have a CSV used for training, rebuild feature_cols from it and reuse.
df_train = pd.read_csv("artifacts/final_train_scaled_binary.csv")  # or your regressor training CSV
TARGET_COL = "MP_binary_high"
exclude = {"SMILES", TARGET_COL, "MW", "MP", "MP_category_default", "Structure_Cluster"}
num_cols = df_train.select_dtypes(include=[np.number]).columns
feature_cols = [c for c in num_cols if c not in exclude]

input_size = len(feature_cols)

# 2) Load models (replace paths with yours)
clf_ckpt      = "artifacts/binary_best_models/binary_best_fold_0.pt"    # change with the best_model
reg_low_ckpt  = "artifacts/lowint_best_models/lowint_best_fold_0.pt"
reg_high_ckpt = "artifacts/high_best_models/high_best_fold_0.pt"

clf_model      = load_model_from_ckpt(clf_ckpt, BinaryClassifier, input_size, device=device)
reg_low_model  = load_model_from_ckpt(reg_low_ckpt, ImprovedNN, input_size, device=device)
reg_high_model = load_model_from_ckpt(reg_high_ckpt, ImprovedNN, input_size, device=device)

# 3) Build one input sample (already scaled example)
# If you start from an unscaled RDKit vector, apply your saved StandardScaler first.
# scaler = joblib.load("artifacts/standard_scaler_train.joblib")
# x_scaled = scaler.transform(x_unscaled.reshape(1,-1)).ravel()

unknown_row = df_train.sample(1, random_state=0)  # demo only
x_scaled = unknown_row[feature_cols].to_numpy(np.float32).ravel()

out = predict_weighted_mp(x_scaled, clf_model, reg_low_model, reg_high_model, device=device)
print(out)
